# Chapter 1 — Claude API Fundamentals · Homework (self-test)

**Goal:** prove you can build, read and control Messages API calls: request structure, roles, all seven `stop_reason` values, system prompts, context and token budgets, sampling and non-determinism, cost and model choice, and calling Claude through other platforms.

- **Theory:** [README.md](README.md) · **Practice:** [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 1.5–2 hours
- **Estimated API cost:** $0. Everything is checked offline; the one optional live cell uses the free token-counting endpoint.
- **Prerequisites:** the practice notebook, and the repo venv from [00 — Prerequisites](../../00-prerequisites/README.md)

### How to use this notebook
1. Attempt everything **without peeking**. Solutions are at the very bottom, inside collapsed blocks.
2. **Part A:** 12 exam-style questions; put your letters in the `answers` cell and run the grader.
3. **Part B:** 7 coding exercises; replace each `TODO` stub, then run its `check_exN()` cell. Checkers never crash the notebook; they print ✓ or a hint.
4. **Part C:** a design scenario; write your answer, then compare with the rubric.
5. **Scorecard:** the pass mark is **72%** on both the quiz and the exercises, which mirrors the Claude Certified Architect exam's 720/1000.

## Setup

In [ ]:
import hashlib
import json
import os
import re
import statistics
import time
from dataclasses import dataclass, field
from types import SimpleNamespace
from typing import Any, Callable

import anthropic
from anthropic.types import Message
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())                       # reads ANTHROPIC_API_KEY (and optional CLAUDE_MODEL) from .env
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
HAIKU = "claude-haiku-4-5-20251001"              # cheapest model; retirement "not sooner than Oct 15, 2026"
client = anthropic.Anthropic()                   # creating the client makes no network call

print("anthropic SDK", anthropic.__version__, "| default model:", MODEL)

In [ ]:
# Only the optional live cell needs a key. Offline cells (mocks, graders, calculators) work without one.
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see ../../00-prerequisites/README.md)"

In [ ]:
# Run this cell; do not edit. Shared helpers for the exercises and the checkers.
def text_of(resp) -> str:
    return "".join(b.text for b in resp.content if b.type == "text")


def fake_message(content: list[dict] | None = None, stop_reason: str = "end_turn", *, model: str = MODEL,
                 stop_sequence: str | None = None, stop_details: dict | None = None,
                 input_tokens: int = 100, output_tokens: int = 20) -> Message:
    """Build a real anthropic.types.Message from plain dicts (offline)."""
    return Message.model_validate({
        "id": "msg_offline", "type": "message", "role": "assistant", "model": model,
        "content": [{"type": "text", "text": "Hello!"}] if content is None else content,
        "stop_reason": stop_reason, "stop_sequence": stop_sequence, "stop_details": stop_details,
        "usage": {"input_tokens": input_tokens, "output_tokens": output_tokens},
    })


class TruncatedError(RuntimeError):
    """The model's output was cut off (max_tokens or model_context_window_exceeded)."""


class RefusedError(RuntimeError):
    """The model declined (stop_reason == 'refusal')."""


PRICES = {   # model: (input $/MTok, output $/MTok, cache-read multiplier), October 2026
    "claude-fable-5-1": (10.00, 50.00, 0.025),
    "claude-opus-5-5": (4.00, 20.00, 0.05),
    "claude-sonnet-5-5": (2.00, 10.00, 0.10),
    HAIKU: (1.00, 5.00, 0.10),
}
CACHE_WRITE_5M = 1.25
CONTEXT_WINDOW = {"claude-fable-5-1": 1_000_000, "claude-opus-5-5": 1_000_000, "claude-sonnet-5-5": 1_000_000, HAIKU: 200_000}

SCORES: dict[str, float] = {}
EX_RESULTS: dict[str, bool] = {f"ex{i}": False for i in range(1, 8)}

# Exercise 7 data (October 2026; the Features overview page is the source of truth)
PLATFORM_CLIENT = {"anthropic": "Anthropic", "bedrock": "AnthropicBedrockMantle",
                   "vertex": "AnthropicVertex", "foundry": "AnthropicFoundry"}
MISSING_FEATURES = {
    "anthropic": set(),
    "bedrock": {"structured_outputs", "message_batches", "models_api", "server_side_fallback",
                "web_search", "web_fetch", "code_execution", "files_api"},
    "vertex": {"message_batches", "models_api", "server_side_fallback", "web_fetch", "code_execution", "files_api"},
    "foundry": {"message_batches", "models_api", "server_side_fallback"},
}


def _record(ex: str, ok: bool, msg: str = "") -> None:
    EX_RESULTS[ex] = ok
    print(f"✓ Exercise {ex[2:]} passed" if ok else f"✗ Exercise {ex[2:]} not passed yet: {msg}")

## Part A — Quiz (12 questions)

Exam style: one scenario, four options, exactly one best answer.

**Q1.** Your summarizer reads `reply = response.content[0].text`. After moving from Haiku 4.5 to Sonnet 5.5, it intermittently raises `AttributeError: 'ThinkingBlock' object has no attribute 'text'`. What is the right fix?

- **A)** Send `thinking={"type": "disabled"}` on every request so the first block is always text
- **B)** Read `response.content[-1].text` instead, because the answer always comes last
- **C)** Iterate over `response.content` and join the `text` of blocks whose `type == "text"`
- **D)** Wrap the line in `try/except AttributeError` and return an empty string

**Q2.** A nightly enrichment job intermittently fails with HTTP 529 `overloaded_error` and 429 `rate_limit_error`. A teammate proposes checking `response.stop_reason == "overloaded"` and string-matching the error text. What should you do instead?

- **A)** Treat them as typed exceptions (`anthropic.OverloadedError`, `anthropic.RateLimitError`, or `APIStatusError` by status), rely on the SDK's built-in backoff retries, tune `max_retries`, and add your own backoff for the job
- **B)** Add `"overloaded"` and `"rate_limited"` cases to the `stop_reason` handler
- **C)** Lower `max_tokens` so each request is small enough to be accepted
- **D)** Catch `Exception`, search the message for "overloaded", and retry immediately in a tight loop

**Q3.** You port an agent from another vendor's API. It sends tool output as `{"role": "tool", "tool_call_id": "...", "content": "..."}` and every request returns 400. How must the tool output be sent to Claude?

- **A)** As a message with `role: "function"` and the tool name
- **B)** Appended to the top-level `system` prompt so the model treats it as trusted
- **C)** As an `assistant` message that contains the result text
- **D)** As a `tool_result` content block inside a `user` message, with `tool_use_id` matching the `tool_use` block

**Q4.** A banking assistant runs on Sonnet 5.5 with a large cached system prompt. Mid-chat, your backend learns the customer's card was just frozen, and the model must treat this as an operator instruction from the next reply on, without invalidating the prompt cache. What do you do?

- **A)** Edit the top-level `system` field to include the event and resend
- **B)** Append `{"role": "system", "content": "Backend event: card frozen ..."}` right after the latest `user` turn, before the model's reply
- **C)** Insert the event as `messages[0]` with `role: "system"`
- **D)** Put the event in a fake `tool_result` block in the next user turn

**Q5.** A research agent uses the server-side web search tool. A response comes back with `stop_reason: "pause_turn"` and a `server_tool_use` block without a result. What should your code do?

- **A)** Append a user message saying "Please continue" and call again
- **B)** Treat it as `end_turn` and return whatever text is present
- **C)** Call again with the same history plus this assistant response appended unchanged as the last turn, with no new user message
- **D)** Raise `max_tokens` and resend the original request without the paused turn

**Q6.** On Sonnet 5.5, dashboards built on HTTP 5xx rates show no errors, but users report occasional blank answers. Your code joins text blocks and returns the result. What is the most likely cause and fix?

- **A)** Some responses are `stop_reason: "refusal"` (HTTP 200, often empty `content`). Branch on `stop_reason` before reading content, show a helpful message or retry on a fallback model, and emit a refusal metric
- **B)** Network timeouts drop the body; raise the client timeout
- **C)** `max_tokens` is too high, so the model returns nothing; lower it
- **D)** When `stop_details` is present the server had an internal error; retry on the same model until text appears

**Q7.** A support bot's system prompt is already concise, uses concrete examples and puts the critical format rule in a tagged `<rules>` section. In 60-turn sessions it still drifts from the format after about turn 40. The model supports mid-conversation system messages. What is the best next step?

- **A)** Switch to a model with a larger context window
- **B)** Move the format rule into the first user message, where it is closer to the conversation
- **C)** Validate every reply and regenerate on failure as the primary control
- **D)** Re-inject a short reminder of the rule at natural breakpoints with a mid-conversation `system` message

**Q8.** To keep long refund conversations affordable, your app summarizes old turns. QA finds the bot now says "around $40" and "next week" where the customer was promised $42.50 by 2026-10-10. What is the best design change?

- **A)** Switch to a 1M-token model and stop summarizing
- **B)** Keep a structured case-facts block (order ID, amount, deadline) that is never summarized, plus a summary of older turns and the recent turns verbatim
- **C)** Keep only the last 10 messages and drop the summary
- **D)** Raise `max_tokens` so the summaries can be longer

**Q9.** Capacity planning for a Sonnet 5.5 migration reused token counts measured earlier on Haiku 4.5 (priced at Sonnet 5.5 rates). After go-live, `usage.input_tokens` is about 30% higher than forecast for identical prompts. What explains it, and what should the team have done?

- **A)** Sonnet 5.5 uses a newer tokenizer that yields roughly 30% more tokens for the same text; recount with `messages.count_tokens` using the target model
- **B)** The token counting endpoint bills its own tokens, which are added to `usage`
- **C)** Prompt caching is disabled by default on Sonnet 5.5, which inflates `input_tokens`
- **D)** Thinking tokens are billed as input tokens on Sonnet 5.5

**Q10.** For a ticket-classification workload, Haiku 4.5 costs $0.40 per 1,000 requests and Sonnet 5.5 at low effort costs $0.75. Haiku's outputs fail validation 40% of the time and need up to 2 extra calls; Sonnet's rarely do. Both meet the latency target. Which metric should decide?

- **A)** Cost per request, so Haiku 4.5
- **B)** The model with the largest context window
- **C)** Cost per completed task at the required quality, measured on a representative eval set
- **D)** Whichever model ranks higher on public benchmarks

**Q11.** An eval of a Sonnet 5.5 ticket classifier ran each of 50 tickets once and scored 88%. Rerunning the unchanged code scored 82%. Product wants one reliable number to decide between two prompt versions. What should you do?

- **A)** Send `temperature=0` through `extra_body` so both runs become identical
- **B)** Run each ticket several times (for example 5) under both prompt versions on the same tickets, compare mean pass rates and their spread, and inspect tickets whose result flips between runs
- **C)** Report the higher of the two runs, because it shows what the prompt can achieve
- **D)** Switch to Opus 5.5, because larger models produce deterministic output

**Q12.** A team moves a Sonnet 5.5 app from the Claude API to Google Cloud with `AnthropicVertex(project_id=..., region="global")`. At startup the app calls `client.models.retrieve(...)` to read the context window, and a nightly job uses the Message Batches API. What breaks, and what should they do?

- **A)** Nothing breaks: Google Cloud exposes every Claude API endpoint
- **B)** Only the model ID must change, to `anthropic.claude-sonnet-5-5`
- **C)** Rewrite the app against a Google-specific request format, because Vertex AI does not accept the Messages API body
- **D)** The Models API and the Message Batches API are not available on Google Cloud: keep model limits in configuration and redesign the nightly job (for example concurrent real-time calls with the async client, or keep that job on the Claude API). The model ID stays `claude-sonnet-5-5`

In [ ]:
# Your answers: "A", "B", "C" or "D"
answers = {"q1": None, "q2": None, "q3": None, "q4": None, "q5": None, "q6": None,
           "q7": None, "q8": None, "q9": None, "q10": None, "q11": None, "q12": None}

In [ ]:
# Answer key stored as sha256(f"{qid}:{letter}") so the letters are not visible
_QUIZ_KEY = {
    "q1": "227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e",
    "q2": "3d9c97ee45947882f5eee5cf878c4f33a0a270d8f2a2e1fc259d66cdc95c7b45",
    "q3": "14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6",
    "q4": "61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa",
    "q5": "e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f",
    "q6": "ac78c6455837b3b074080e7912c1bdab72fbbfdcaeefd9269fe293a50c4a9522",
    "q7": "45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457",
    "q8": "39f95f59afaa9522616af26bd34df1e0184030ae8f738e319fc9a1f15b075834",
    "q9": "8b77cfcdcf7bee660b161787fd482af13fe960d0da61692597e6e870d4668936",
    "q10": "d6ea6e76ddfa07a3b78a8bd000db76a223a8ccc3f38e290cf5a68d95e5494cc5",
    "q11": "42310be80adadfed411c1783b5c42432624ffc39d2ea2f7d837996726d6b0774",
    "q12": "33d12c43f8f028fb844551369083e2b9daa804b371ce6b8835f63a07a5cac42e",
}


def grade_quiz(answers: dict) -> float:
    correct = 0
    for qid, digest in _QUIZ_KEY.items():
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else ""
        ok = bool(letter) and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == digest
        correct += ok
        print(f"{qid:>4}: {'✓' if ok else '✗'}{'' if letter else '  (unanswered)'}")
    score = correct / len(_QUIZ_KEY)
    SCORES["quiz"] = score
    print(f"\nQuiz: {correct}/{len(_QUIZ_KEY)} = {score:.0%}")
    return score

quiz_score = grade_quiz(answers)

## Part B — Coding exercises

All seven are checked **offline** with mock `Message` objects, fake clients or plain data. Edit only the `TODO` cells.

### Exercise 1 — Build a correct request (1.1)

Implement `build_request(...)` so it returns the keyword arguments for `client.messages.create(**kwargs)`:

- always: `model`, `max_tokens`, and `messages` with exactly one `user` turn holding `question` (string content)
- `system` only when given, as a **top-level** field (never a message)
- `stop_sequences` only when the list is non-empty
- `effort` only when given, as `output_config={"effort": effort}`; raise `ValueError` for values other than `low`, `medium`, `high`, `xhigh`, `max`, and for Haiku models (they do not support effort)
- `end_user_id` only when given, as `metadata={"user_id": <sha256 hex digest of end_user_id>}` (an opaque ID, never the raw email)
- raise `ValueError` if `max_tokens` is not a positive `int`

In [ ]:
def build_request(question: str, *, model: str = MODEL, system: str | None = None, max_tokens: int = 1024,
                  stop_sequences: list[str] | None = None, effort: str | None = None,
                  end_user_id: str | None = None) -> dict:
    """Return the kwargs for client.messages.create(**kwargs)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    try:
        r = build_request("Hi", model="claude-sonnet-5-5")
        assert r == {"model": "claude-sonnet-5-5", "max_tokens": 1024,
                     "messages": [{"role": "user", "content": "Hi"}]}, f"minimal request wrong: {r}"
        r = build_request("Q", system="Be brief.", stop_sequences=["</answer>"], effort="low", end_user_id="ana@example.com")
        assert r.get("system") == "Be brief.", "system must be a top-level field"
        assert all(m["role"] != "system" for m in r["messages"]), "do not put the system prompt in messages"
        assert r.get("stop_sequences") == ["</answer>"], "stop_sequences missing"
        assert r.get("output_config") == {"effort": "low"}, "effort goes in output_config={'effort': ...}"
        uid = r.get("metadata", {}).get("user_id", "")
        assert uid == hashlib.sha256(b"ana@example.com").hexdigest(), "metadata.user_id must be the sha256 hex of the id"
        assert "stop_sequences" not in build_request("Q", stop_sequences=[]), "omit empty stop_sequences"
        for bad in (dict(effort="turbo"), dict(model=HAIKU, effort="low"), dict(max_tokens=0)):
            try:
                build_request("Q", **bad)
            except ValueError:
                pass
            else:
                raise AssertionError(f"expected ValueError for {bad}")
        import inspect
        inspect.signature(client.messages.create).bind(**r)    # every key is a real SDK parameter
        _record("ex1", True)
    except NotImplementedError:
        _record("ex1", False, "build_request is not implemented yet")
    except Exception as e:
        _record("ex1", False, f"{type(e).__name__}: {e}")


check_ex1()

### Exercise 2 — Read a response safely (1.1, 1.3)

Implement `summarize_response(resp)` for an `anthropic.types.Message`. Return a dict with exactly these keys:

- `"text"`: all `text` blocks joined in order (`""` if none). Skip thinking and tool blocks.
- `"tool_calls"`: a list of `(id, name, input)` tuples for every `tool_use` block, in order
- `"truncated"`: `True` for `max_tokens` **and** `model_context_window_exceeded`
- `"refused"`: `True` when `stop_reason == "refusal"`
- `"refusal_category"`: `stop_details.category` for refusals (it can be `None`), else `None`

In [ ]:
def summarize_response(resp) -> dict:
    """Return {"text", "tool_calls", "truncated", "refused", "refusal_category"} for a Message."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    try:
        r = summarize_response(fake_message([
            {"type": "thinking", "thinking": "", "signature": "s"},
            {"type": "text", "text": "Checking "},
            {"type": "tool_use", "id": "toolu_1", "name": "get_order", "input": {"order_id": "A17"}},
            {"type": "text", "text": "now."}], "tool_use"))
        assert r["text"] == "Checking now.", f"text should join text blocks only: {r['text']!r}"
        assert r["tool_calls"] == [("toolu_1", "get_order", {"order_id": "A17"})], f"tool_calls wrong: {r['tool_calls']}"
        assert r["truncated"] is False and r["refused"] is False and r["refusal_category"] is None
        r = summarize_response(fake_message([], "refusal", stop_details={"type": "refusal", "category": "bio", "explanation": None}))
        assert r == {"text": "", "tool_calls": [], "truncated": False, "refused": True, "refusal_category": "bio"}, r
        r = summarize_response(fake_message([], "refusal", stop_details={"type": "refusal", "category": None, "explanation": None}))
        assert r["refused"] is True and r["refusal_category"] is None, "category can be None on a refusal"
        for sr in ("max_tokens", "model_context_window_exceeded"):
            assert summarize_response(fake_message([{"type": "text", "text": "cut"}], sr))["truncated"] is True, sr
        assert summarize_response(fake_message(stop_reason="stop_sequence", stop_sequence="END"))["truncated"] is False
        _record("ex2", True)
    except NotImplementedError:
        _record("ex2", False, "summarize_response is not implemented yet")
    except Exception as e:
        _record("ex2", False, f"{type(e).__name__}: {e}")


check_ex2()

### Exercise 3 — Agent-loop control flow on fake responses (1.2, 1.3)

Implement `run_agent_turn(create, messages, tools, max_steps=6)`. `create(messages)` returns a `Message`; `tools` maps tool names to Python functions. Mutate `messages` in place:

- `end_turn` or `stop_sequence`: append the assistant turn (`resp.content`) and return its text
- `tool_use`: append the assistant turn, run **every** `tool_use` block (`tools[name](**input)`), then append **one** `user` message whose content is all the `tool_result` blocks (`tool_use_id`, `content=str(result)`). An unknown tool or an exception becomes a `tool_result` with `"is_error": True`. Loop.
- `pause_turn`: append the assistant turn unchanged and loop (no new user message)
- `max_tokens` / `model_context_window_exceeded`: raise `TruncatedError` (never run tools from that turn)
- `refusal`: raise `RefusedError`
- more than `max_steps` calls: raise `RuntimeError`

In [ ]:
def run_agent_turn(create: Callable[[list[dict]], Any], messages: list[dict],
                   tools: dict[str, Callable[..., Any]], max_steps: int = 6) -> str:
    """Drive one user turn to completion. `create(messages)` returns a Message. Mutates `messages`."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    def scripted(*responses):
        queue, seen = list(responses), []
        def create(messages):
            seen.append([dict(m) for m in messages])
            return queue.pop(0)
        return create, seen

    two_tools = fake_message([
        {"type": "tool_use", "id": "t1", "name": "get_order", "input": {"order_id": "A17"}},
        {"type": "tool_use", "id": "t2", "name": "get_weather", "input": {"city": "Almaty"}}], "tool_use")
    final = fake_message([{"type": "text", "text": "All set."}])
    try:
        calls = []
        tools = {"get_order": lambda order_id: calls.append(order_id) or "shipped",
                 "get_weather": lambda city: calls.append(city) or "sunny"}
        msgs = [{"role": "user", "content": "Order and weather?"}]
        create, seen = scripted(two_tools, final)
        out = run_agent_turn(create, msgs, tools)
        assert out == "All set.", f"should return the final text, got {out!r}"
        assert calls == ["A17", "Almaty"], f"run EVERY tool_use block, got {calls}"
        tr = seen[1][-1]
        assert tr["role"] == "user" and len(tr["content"]) == 2, "send ALL tool_results in ONE user message"
        assert {b["tool_use_id"] for b in tr["content"]} == {"t1", "t2"}, "match tool_use_id to each call"
        assert msgs[-1]["role"] == "assistant", "append the final assistant turn to messages"

        create, seen = scripted(fake_message([{"type": "tool_use", "id": "t9", "name": "nope", "input": {}}], "tool_use"), final)
        run_agent_turn(create, [{"role": "user", "content": "x"}], tools)
        assert seen[1][-1]["content"][0].get("is_error") is True, "unknown/failed tool -> tool_result with is_error=True"

        paused = fake_message([{"type": "server_tool_use", "id": "s1", "name": "web_search", "input": {"query": "q"}}], "pause_turn")
        create, seen = scripted(paused, final)
        run_agent_turn(create, [{"role": "user", "content": "search"}], tools)
        assert [m["role"] for m in seen[1]] == ["user", "assistant"], "pause_turn: resend unchanged, no new user message"

        half = fake_message([{"type": "tool_use", "id": "t3", "name": "get_order", "input": {}}], "max_tokens")
        calls.clear()
        create, _ = scripted(half)
        try:
            run_agent_turn(create, [{"role": "user", "content": "x"}], tools)
            raise AssertionError("max_tokens must raise TruncatedError")
        except TruncatedError:
            assert calls == [], "never run a tool from a truncated turn"
        create, _ = scripted(fake_message([], "refusal", stop_details={"type": "refusal", "category": "cyber", "explanation": None}))
        try:
            run_agent_turn(create, [{"role": "user", "content": "x"}], tools)
            raise AssertionError("refusal must raise RefusedError")
        except RefusedError:
            pass
        loop = fake_message([{"type": "tool_use", "id": "t4", "name": "get_order", "input": {"order_id": "B1"}}], "tool_use")
        create, _ = scripted(*[loop] * 10)
        try:
            run_agent_turn(create, [{"role": "user", "content": "x"}], tools, max_steps=3)
            raise AssertionError("more than max_steps calls must raise RuntimeError")
        except (TruncatedError, RefusedError):
            raise AssertionError("max_steps should raise RuntimeError")
        except RuntimeError:
            pass
        _record("ex3", True)
    except NotImplementedError:
        _record("ex3", False, "run_agent_turn is not implemented yet")
    except Exception as e:
        _record("ex3", False, f"{type(e).__name__}: {e}")


check_ex3()

### Exercise 4 — Validate a `messages` array before sending (1.2)

Implement `validate_messages(messages)` returning a list of problem codes (order does not matter; `[]` = valid):

| Code | When |
|---|---|
| `"empty"` | the list is empty (return only this) |
| `"first_not_user"` | `messages[0]` is not a `user` turn |
| `"bad_role:{i}"` | role is not `user`, `assistant` or `system` (e.g. `"tool"`) |
| `"tool_result_not_in_user:{i}"` | a non-`user` message contains a `tool_result` block |
| `"system_after_non_user:{i}"` | a `system` message (at `i > 0`) does not directly follow a `user` turn |
| `"system_before_non_assistant:{i}"` | a `system` message (at `i > 0`) is followed by something other than an `assistant` turn (being last is fine) |
| `"prefill:{i}"` | the array has more than one message and ends with an `assistant` turn (final-turn prefill: 400 on current models) |

For this exercise, ignore the server-tool exception (a system message may also follow an assistant turn that ends in a server tool result).

In [ ]:
def validate_messages(messages: list[dict]) -> list[str]:
    """Return a list of problem codes (empty list = valid)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    U = lambda t="hi": {"role": "user", "content": t}
    A = lambda t="ok": {"role": "assistant", "content": t}
    S = lambda t="note": {"role": "system", "content": t}
    cases = [
        ([], ["empty"]),
        ([U(), A(), U()], []),
        ([U(), A(), U(), S()], []),
        ([U(), S(), A(), U()], []),
        ([S(), U()], ["first_not_user"]),
        ([U(), A(), S()], ["system_after_non_user:2"]),
        ([U(), A(), S(), U()], ["system_after_non_user:2", "system_before_non_assistant:2"]),
        ([U(), S(), U()], ["system_before_non_assistant:1"]),
        ([U(), {"role": "tool", "content": "42"}], ["bad_role:1"]),
        ([U(), {"role": "assistant", "content": [{"type": "tool_result", "tool_use_id": "t1", "content": "x"}]}, U()],
         ["tool_result_not_in_user:1"]),
        ([U(), A("{")], ["prefill:1"]),
    ]
    try:
        for msgs, expected in cases:
            got = validate_messages(msgs)
            assert sorted(got or []) == sorted(expected), f"{[m['role'] for m in msgs]}: expected {expected}, got {got}"
        _record("ex4", True)
    except NotImplementedError:
        _record("ex4", False, "validate_messages is not implemented yet")
    except Exception as e:
        _record("ex4", False, f"{type(e).__name__}: {e}")


check_ex4()

### Exercise 5 — Cost per request and per completed task (1.6, 1.7)

1. `estimate_cost(usage, model)`: use `PRICES` and `CACHE_WRITE_5M` from the helper cell. Cost = (`input_tokens` × input price + `cache_creation_input_tokens` × input price × 1.25 + `cache_read_input_tokens` × input price × cache-read multiplier + `output_tokens` × output price) / 1,000,000. Treat `None` cache fields as 0. Raise `ValueError` for a model not in `PRICES`.
2. `pick_model(rows, min_accuracy, max_p50_ms)`: each row has `name`, `accuracy`, `p50_ms`, `cost_per_request`, `avg_calls_per_task`. Among rows meeting **both** targets, return the `name` with the lowest **cost per completed task** (`cost_per_request × avg_calls_per_task`); `None` if none pass.

In [ ]:
def estimate_cost(usage, model: str) -> float:
    """Dollar cost of one response's usage (5-minute cache writes)."""
    # TODO
    raise NotImplementedError


def pick_model(rows: list[dict], min_accuracy: float, max_p50_ms: float) -> str | None:
    """Name of the cheapest row by cost per COMPLETED task that meets both targets, else None."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    try:
        U = lambda i, o, cw=None, cr=None: SimpleNamespace(input_tokens=i, output_tokens=o,
                                                            cache_creation_input_tokens=cw, cache_read_input_tokens=cr)
        close = lambda a, b: abs(a - b) < 1e-9
        assert close(estimate_cost(U(1_000_000, 0), "claude-sonnet-5-5"), 2.0), "1M input tokens on Sonnet 5.5 = $2"
        assert close(estimate_cost(U(0, 1_000_000), HAIKU), 5.0), "1M output tokens on Haiku 4.5 = $5"
        assert close(estimate_cost(U(1000, 200), "claude-sonnet-5-5"), 0.004), "None cache fields count as 0"
        assert close(estimate_cost(U(0, 0, cw=1_000_000), "claude-sonnet-5-5"), 2.5), "5-minute cache write = 1.25x input"
        assert close(estimate_cost(U(0, 0, cr=1_000_000), "claude-opus-5-5"), 0.2), "cache read on Opus 5.5 = 0.05x input"
        assert close(estimate_cost(U(0, 0, cr=1_000_000), "claude-fable-5-1"), 0.25), "cache read on Fable 5.1 = 0.025x input"
        try:
            estimate_cost(U(1, 1), "gpt-unknown")
            raise AssertionError("unknown model must raise ValueError")
        except ValueError:
            pass
        rows = [
            {"name": "haiku", "accuracy": 0.93, "p50_ms": 600, "cost_per_request": 0.0004, "avg_calls_per_task": 2.2},
            {"name": "sonnet-low", "accuracy": 0.95, "p50_ms": 1100, "cost_per_request": 0.00075, "avg_calls_per_task": 1.0},
            {"name": "opus", "accuracy": 0.97, "p50_ms": 2500, "cost_per_request": 0.0015, "avg_calls_per_task": 1.0},
        ]
        assert pick_model(rows, 0.90, 2000) == "sonnet-low", "compare cost per COMPLETED task (cost x calls per task)"
        assert pick_model(rows, 0.96, 3000) == "opus", "respect the accuracy bar"
        assert pick_model(rows, 0.99, 3000) is None, "return None when nothing passes"
        _record("ex5", True)
    except NotImplementedError:
        _record("ex5", False, "estimate_cost / pick_model not implemented yet")
    except Exception as e:
        _record("ex5", False, f"{type(e).__name__}: {e}")


check_ex5()

### Exercise 6 — Will it fit? Count before you send (1.5, 1.6)

Implement `fits_in_window(client, model, system, messages, max_tokens)`:

- call `client.messages.count_tokens(model=model, system=system, messages=messages)` (same model you will call) and read `.input_tokens`
- `CONTEXT_WINDOW[model]` gives the window
- return `"too_long"` if the input alone exceeds the window (the API would return 400), `"may_hit_window"` if input + `max_tokens` exceeds it (accepted, but may stop with `model_context_window_exceeded`), else `"ok"`

The checker uses a fake client, so it is offline. An optional live run follows **(live — calls the free `count_tokens` endpoint, $0)**.

In [ ]:
def fits_in_window(client, model: str, system: str, messages: list[dict], max_tokens: int) -> str:
    """Return "too_long", "may_hit_window" or "ok" using client.messages.count_tokens."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex6() -> None:
    class FakeCounter:
        def __init__(self, n):
            self.n, self.calls = n, []
            self.messages = SimpleNamespace(count_tokens=self._count)
        def _count(self, **kwargs):
            self.calls.append(kwargs)
            return SimpleNamespace(input_tokens=self.n)
    msgs = [{"role": "user", "content": "contract text ..."}]
    try:
        cases = [(5_000, "claude-sonnet-5-5", 16_000, "ok"), (990_000, "claude-sonnet-5-5", 16_000, "may_hit_window"),
                 (1_000_001, "claude-sonnet-5-5", 1_000, "too_long"), (190_000, HAIKU, 16_000, "may_hit_window"),
                 (250_000, HAIKU, 1_000, "too_long")]
        for n, model, mx, expected in cases:
            fake = FakeCounter(n)
            got = fits_in_window(fake, model, "Be brief.", msgs, mx)
            assert got == expected, f"{n:,} tokens on {model} with max_tokens={mx:,}: expected {expected!r}, got {got!r}"
            sent = fake.calls[0] if fake.calls else {}
            assert sent.get("model") == model, "count with the SAME model you will call"
            assert sent.get("system") == "Be brief." and sent.get("messages") == msgs, "pass system and messages to count_tokens"
        _record("ex6", True)
    except NotImplementedError:
        _record("ex6", False, "fits_in_window is not implemented yet")
    except Exception as e:
        _record("ex6", False, f"{type(e).__name__}: {e}")


check_ex6()

In [ ]:
# [live] Optional, free: the count_tokens endpoint is not billed
try:
    doc = "Clause 1. The supplier shall deliver ... " * 2000
    print(fits_in_window(client, MODEL, "You review contracts.", [{"role": "user", "content": doc}], max_tokens=16_000))
except NotImplementedError:
    print("Implement fits_in_window first.")
except anthropic.APIError as e:
    print(f"Live call failed ({type(e).__name__}); check your key in .env.")

### Exercise 7 — Plan the same call on another platform (1.8)

Implement `plan_platform_call(platform, model, features, deployment=None)` for `platform` in `"anthropic"`, `"bedrock"`, `"vertex"`, `"foundry"` (raise `ValueError` for anything else). `model` is a first-party ID such as `claude-sonnet-5-5` or the dated `claude-haiku-4-5-20251001`. Return a dict:

- `"client"`: the Python client class name from `PLATFORM_CLIENT` (helper cell)
- `"model"`: the model ID for that platform:
  - `anthropic`: unchanged
  - `bedrock`: `"anthropic."` + the ID **without** a trailing `-YYYYMMDD` date (`anthropic.claude-haiku-4-5`)
  - `vertex`: unchanged, except a dated ID uses `@` before the date (`claude-haiku-4-5@20251001`)
  - `foundry`: `deployment` if given, else the ID without a trailing date
- `"missing"`: the sorted list of requested `features` that appear in `MISSING_FEATURES[platform]`

In [ ]:
def plan_platform_call(platform: str, model: str, features: set[str], deployment: str | None = None) -> dict:
    """Return {"client": <class name>, "model": <model ID for that platform>, "missing": <sorted list>}."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex7() -> None:
    cases = [
        (("bedrock", "claude-sonnet-5-5", {"structured_outputs", "prompt_caching"}),
         {"client": "AnthropicBedrockMantle", "model": "anthropic.claude-sonnet-5-5", "missing": ["structured_outputs"]}),
        (("bedrock", "claude-haiku-4-5-20251001", set()),
         {"client": "AnthropicBedrockMantle", "model": "anthropic.claude-haiku-4-5", "missing": []}),
        (("vertex", "claude-haiku-4-5-20251001", {"message_batches", "web_search"}),
         {"client": "AnthropicVertex", "model": "claude-haiku-4-5@20251001", "missing": ["message_batches"]}),
        (("vertex", "claude-sonnet-5-5", set()),
         {"client": "AnthropicVertex", "model": "claude-sonnet-5-5", "missing": []}),
        (("foundry", "claude-opus-5-5", {"server_side_fallback", "models_api", "structured_outputs"}, "opus-prod"),
         {"client": "AnthropicFoundry", "model": "opus-prod", "missing": ["models_api", "server_side_fallback"]}),
        (("foundry", "claude-opus-5-5", set()),
         {"client": "AnthropicFoundry", "model": "claude-opus-5-5", "missing": []}),
        (("anthropic", "claude-haiku-4-5-20251001", {"message_batches", "server_side_fallback"}),
         {"client": "Anthropic", "model": "claude-haiku-4-5-20251001", "missing": []}),
    ]
    try:
        for args, expected in cases:
            got = plan_platform_call(*args)
            assert isinstance(got, dict), f"return a dict, got {got!r}"
            for key in ("client", "model", "missing"):
                assert got.get(key) == expected[key], f"plan_platform_call{args}: {key} expected {expected[key]!r}, got {got.get(key)!r}"
        try:
            plan_platform_call("azure-openai", "claude-sonnet-5-5", set())
            raise AssertionError("an unknown platform must raise ValueError")
        except ValueError:
            pass
        _record("ex7", True)
    except NotImplementedError:
        _record("ex7", False, "plan_platform_call is not implemented yet")
    except Exception as e:
        _record("ex7", False, f"{type(e).__name__}: {e}")


check_ex7()

## Part C — Architecture scenario

**Acme Cloud support chat.** Acme wants a customer-support chat on the Claude API for 200,000 customers. Sessions can run 60+ turns. Agents must remember order IDs, refund amounts and promised deadlines exactly. A few tools exist (`get_customer`, `get_order`, `issue_refund`), and some answers need the server-side web search tool. Target: p95 under 4 s for simple questions, a monthly budget cap, and clean handling of classifier refusals. The default model today is Sonnet 5.5.

Describe your design in 200–400 words. Cover: where conversation state lives and how you keep it within budget; how your loop handles each `stop_reason`; what goes in the system prompt vs. mid-conversation messages; how you measure tokens and cost; and how you would choose the model and effort level(s).

**Your answer:**

*(double-click to edit)*

<details><summary>Rubric: what a strong answer contains (open after writing yours)</summary>

- **State:** the app owns history (the API is stateless); persisted per session in your store; full content lists (incl. thinking/tool blocks) are replayed; no reliance on a "session" feature.
- **Context budget:** hybrid history: an un-summarized **case-facts** block (order IDs, amounts, deadlines), a summary of old turns, recent turns verbatim; tool outputs trimmed to needed fields; prompt caching on the stable prefix (system + tools); `count_tokens` before risky requests; compaction mentioned as an option.
- **Loop:** continue on `tool_use` (run all tools, one `user` turn of `tool_result`s, `is_error` on failures); resend unchanged on `pause_turn` (web search); `max_tokens` → raise budget / continue, never execute a truncated `tool_use`; `model_context_window_exceeded` → shrink input, not raise `max_tokens`; `refusal` → check before reading content, `stop_details` for the category, server-side fallback (beta) or a fallback model, refusal metrics (they are HTTP 200); unknown values fail safe; an iteration cap as a safety net only.
- **Errors:** typed SDK exceptions, SDK retries for 429/5xx/529, idempotency for `issue_refund` so a retry cannot refund twice.
- **Prompting:** persona, scope and format in the top-level system prompt; scoped tool rules ("before acting on a specific order…") instead of "always call get_customer"; backend events via mid-conversation `system` messages (supported on Sonnet 5.5) so the cache survives; untrusted tool/web text never placed in system messages.
- **Cost and model choice:** usage logged per call and per session; cost per **resolved conversation**; an eval set from real tickets; try Sonnet 5.5 at lower effort and Haiku 4.5 for routing/classification (note its near retirement date), Opus 5.5 only where evals demand; keep the model ID in config; monitor budget and alert.
</details>

## Scorecard

In [ ]:
quiz = SCORES.get("quiz")
passed = sum(EX_RESULTS.values())
ex_score = passed / len(EX_RESULTS)
print(f"Quiz:      {'not graded yet' if quiz is None else f'{quiz:.0%}'}")
print(f"Exercises: {passed}/{len(EX_RESULTS)} passed = {ex_score:.0%}  " +
      " ".join(f"{k}:{'✓' if v else '✗'}" for k, v in EX_RESULTS.items()))
PASS_MARK = 0.72
ok = quiz is not None and quiz >= PASS_MARK and ex_score >= PASS_MARK
print("\nRESULT:", "PASS ✓ (both parts >= 72%)" if ok else "RETRY: reach 72% on the quiz AND the exercises (Part C is self-assessed)")

**Next chapter:** [Chapter 2 — Tool Use](../02-tool-use/README.md)

---

## Solutions (spoilers)

Open only after your own attempt.

<details><summary>Quiz answers and explanations</summary>

- **Q1: C.** Content is a list of typed blocks; filter by type. A fails on Sonnet 5.5 (`disabled` returns 400; the off switch there is `between_tools`), and it treats a symptom. B breaks on tool calls and empty refusals. D silently drops answers.
- **Q2: A.** Failed requests never produce a `stop_reason`; they raise typed exceptions. The SDK already retries 408/409/429/5xx twice with exponential backoff. B confuses errors with stop reasons; C does not address capacity; D string-matches and hammers the API.
- **Q3: D.** Claude has no `tool` role. Tool results are `tool_result` blocks in a `user` turn, linked by `tool_use_id`. B would also give untrusted tool output operator authority.
- **Q4: B.** Mid-conversation system messages (supported on Sonnet 5.5) keep the cached prefix intact and carry operator authority; they must follow a user turn and precede an assistant turn or end the array. A invalidates the cache, C is invalid (a system message with content can't be `messages[0]`), D has no matching `tool_use` and no operator authority.
- **Q5: C.** `pause_turn` means the server-side tool loop hit its iteration limit; you continue by sending the assistant content back as is. A adds an unnecessary turn, B returns an unfinished answer, D throws away the work and misdiagnoses the cause.
- **Q6: A.** Classifier refusals are successful responses, so error-rate monitoring never sees them. `stop_details` is populated only for refusals. Retrying the same model (D) usually earns another refusal; the docs recommend a fallback model.
- **Q7: D.** With the cheaper steps (concrete prompt, tagged critical rules) done, short reminders are next. Validation and regeneration (C) are a corrective last resort that adds latency. A does not address drift; B gives the rule less authority.
- **Q8: B.** Summarization tends to blur exact values; a separate facts block preserves them while the rest is compressed. A raises cost and latency without fixing curation, C silently drops the facts, D does not stop values from being paraphrased.
- **Q9: A.** Claude 4.7 and later models use a newer tokenizer (about 30% more tokens; content-dependent). Counts are model-specific, so count with the model you will call. Counting is free (B), caching does not change the tokenizer (C), and thinking is billed as output (D).
- **Q10: C.** A cheap model that needs retries or extra turns is not cheap. Measure cost per completed task at the quality bar on your own data. Context size (B) is irrelevant for short tickets; public benchmarks (D) are not your workload.
- **Q11: B.** Sampling and adaptive thinking make identical requests vary, so a single run per ticket is a noisy measurement. Several runs per example give a stable pass rate and expose flaky tickets. A returns 400 on Sonnet 5.5 (and temperature 0 never guaranteed identical output). C is cherry-picking. D is false: no current model is deterministic.
- **Q12: D.** Google Cloud serves the same Messages API body (the model moves into the URL and `anthropic_version` into the body, which the client handles), but it lacks the Models API and Message Batches. A is false; B is the Bedrock ID format; C confuses the transport details the client hides with a different API.
</details>

<details><summary>Exercise 1: <code>build_request</code></summary>

```python
def build_request(question: str, *, model: str = MODEL, system: str | None = None, max_tokens: int = 1024,
                  stop_sequences: list[str] | None = None, effort: str | None = None,
                  end_user_id: str | None = None) -> dict:
    if not isinstance(max_tokens, int) or max_tokens <= 0:
        raise ValueError("max_tokens must be a positive int")
    req: dict = {"model": model, "max_tokens": max_tokens,
                 "messages": [{"role": "user", "content": question}]}
    if system:
        req["system"] = system                      # top level, never inside messages
    if stop_sequences:
        req["stop_sequences"] = list(stop_sequences)
    if effort is not None:
        if model.startswith("claude-haiku"):
            raise ValueError(f"{model} does not support effort")
        if effort not in ("low", "medium", "high", "xhigh", "max"):
            raise ValueError(f"unknown effort {effort!r}")
        req["output_config"] = {"effort": effort}
    if end_user_id:
        req["metadata"] = {"user_id": hashlib.sha256(end_user_id.encode()).hexdigest()}
    return req
```
</details>

<details><summary>Exercise 2: <code>summarize_response</code></summary>

```python
def summarize_response(resp) -> dict:
    details = resp.stop_details if resp.stop_reason == "refusal" else None
    return {
        "text": "".join(b.text for b in resp.content if b.type == "text"),
        "tool_calls": [(b.id, b.name, b.input) for b in resp.content if b.type == "tool_use"],
        "truncated": resp.stop_reason in ("max_tokens", "model_context_window_exceeded"),
        "refused": resp.stop_reason == "refusal",
        "refusal_category": details.category if details else None,
    }
```
</details>

<details><summary>Exercise 3: <code>run_agent_turn</code></summary>

```python
def run_agent_turn(create: Callable[[list[dict]], Any], messages: list[dict],
                   tools: dict[str, Callable[..., Any]], max_steps: int = 6) -> str:
    for _ in range(max_steps):
        resp = create(messages)
        match resp.stop_reason:
            case "end_turn" | "stop_sequence":
                messages.append({"role": "assistant", "content": resp.content})
                return text_of(resp)
            case "tool_use":
                messages.append({"role": "assistant", "content": resp.content})
                results = []
                for b in resp.content:
                    if b.type != "tool_use":
                        continue
                    try:
                        results.append({"type": "tool_result", "tool_use_id": b.id,
                                        "content": str(tools[b.name](**b.input))})
                    except Exception as e:
                        results.append({"type": "tool_result", "tool_use_id": b.id,
                                        "content": f"{type(e).__name__}: {e}", "is_error": True})
                messages.append({"role": "user", "content": results})      # ONE user turn
            case "pause_turn":
                messages.append({"role": "assistant", "content": resp.content})  # resend unchanged
            case "max_tokens" | "model_context_window_exceeded":
                raise TruncatedError(resp.stop_reason)
            case "refusal":
                category = resp.stop_details.category if resp.stop_details else None
                raise RefusedError(str(category))
            case other:
                raise RuntimeError(f"unknown stop_reason {other!r}")
    raise RuntimeError("max_steps exceeded")
```
</details>

<details><summary>Exercise 4: <code>validate_messages</code></summary>

```python
def validate_messages(messages: list[dict]) -> list[str]:
    if not messages:
        return ["empty"]
    problems = []
    if messages[0].get("role") != "user":
        problems.append("first_not_user")
    for i, m in enumerate(messages):
        role = m.get("role")
        if role not in ("user", "assistant", "system"):
            problems.append(f"bad_role:{i}")
            continue
        blocks = m["content"] if isinstance(m.get("content"), list) else []
        if role != "user" and any(b.get("type") == "tool_result" for b in blocks):
            problems.append(f"tool_result_not_in_user:{i}")
        if role == "system" and i > 0:
            if messages[i - 1].get("role") != "user":
                problems.append(f"system_after_non_user:{i}")
            if i + 1 < len(messages) and messages[i + 1].get("role") != "assistant":
                problems.append(f"system_before_non_assistant:{i}")
    if len(messages) > 1 and messages[-1].get("role") == "assistant":
        problems.append(f"prefill:{len(messages) - 1}")
    return problems
```
</details>

<details><summary>Exercise 5: <code>estimate_cost / pick_model</code></summary>

```python
def estimate_cost(usage, model: str) -> float:
    if model not in PRICES:
        raise ValueError(f"no price for {model}")
    inp, out, read_mult = PRICES[model]
    cache_write = usage.cache_creation_input_tokens or 0
    cache_read = usage.cache_read_input_tokens or 0
    return (usage.input_tokens * inp + cache_write * inp * CACHE_WRITE_5M
            + cache_read * inp * read_mult + usage.output_tokens * out) / 1e6


def pick_model(rows: list[dict], min_accuracy: float, max_p50_ms: float) -> str | None:
    ok = [r for r in rows if r["accuracy"] >= min_accuracy and r["p50_ms"] <= max_p50_ms]
    if not ok:
        return None
    return min(ok, key=lambda r: r["cost_per_request"] * r["avg_calls_per_task"])["name"]
```
</details>

<details><summary>Exercise 6: <code>fits_in_window</code></summary>

```python
def fits_in_window(client, model: str, system: str, messages: list[dict], max_tokens: int) -> str:
    n = client.messages.count_tokens(model=model, system=system, messages=messages).input_tokens
    window = CONTEXT_WINDOW[model]
    if n > window:
        return "too_long"
    if n + max_tokens > window:
        return "may_hit_window"
    return "ok"
```
</details>

<details><summary>Exercise 7: <code>plan_platform_call</code></summary>

```python
def plan_platform_call(platform: str, model: str, features: set[str], deployment: str | None = None) -> dict:
    if platform not in PLATFORM_CLIENT:
        raise ValueError(f"unknown platform {platform!r}")
    dated = re.fullmatch(r"(.+)-(\d{8})", model)       # e.g. claude-haiku-4-5-20251001
    base = dated.group(1) if dated else model
    if platform == "anthropic":
        model_id = model
    elif platform == "bedrock":
        model_id = "anthropic." + base
    elif platform == "vertex":
        model_id = f"{dated.group(1)}@{dated.group(2)}" if dated else model
    else:                                                 # foundry: the deployment name
        model_id = deployment or base
    return {"client": PLATFORM_CLIENT[platform], "model": model_id,
            "missing": sorted(set(features) & MISSING_FEATURES[platform])}
```
</details>